# EDA + Drift Analysis Walkthrough

This notebook walks through exploratory data analysis on the reference and production datasets,
runs drift checks, and visualises the results.

**Prerequisites:** run `python model/train.py` and `python data/generate_production_data.py` first.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load datasets

In [ ]:
ref  = pd.read_csv('../data/reference_data.csv')
prod = pd.read_csv('../data/production_data.csv')

print('Reference shape :', ref.shape)
print('Production shape:', prod.shape)
ref.head()

## 2. Summary statistics

In [ ]:
comparison = pd.concat(
    [ref.describe().T.add_suffix('_ref'), prod.describe().T.add_suffix('_prod')],
    axis=1
)[['mean_ref', 'mean_prod', 'std_ref', 'std_prod']]
comparison['mean_delta'] = comparison['mean_prod'] - comparison['mean_ref']
comparison

## 3. Distribution comparison

In [ ]:
numeric_cols = ref.select_dtypes(include=np.number).columns.tolist()
n = len(numeric_cols)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
if n == 1:
    axes = [axes]

for ax, col in zip(axes, numeric_cols):
    ax.hist(ref[col],  bins=20, alpha=0.6, label='Reference',  color='steelblue', density=True)
    ax.hist(prod[col], bins=20, alpha=0.6, label='Production', color='tomato',    density=True)
    ax.set_title(col)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 4. Run drift detection

In [ ]:
from monitoring.drift_detector import run_drift_check

drift_report = run_drift_check(
    '../data/reference_data.csv',
    '../data/production_data.csv',
    psi_threshold=0.1,
    ks_p_threshold=0.05,
)

drift_df = pd.DataFrame([
    {'feature': feat, **metrics}
    for feat, metrics in drift_report.items()
])
drift_df

## 5. PSI bar chart

In [ ]:
if 'psi' in drift_df.columns:
    colors = ['tomato' if d else 'steelblue' for d in drift_df['drift_detected']]
    plt.figure(figsize=(max(6, len(drift_df) * 1.5), 4))
    plt.bar(drift_df['feature'], drift_df['psi'], color=colors)
    plt.axhline(0.1, color='orange', linestyle='--', label='PSI threshold (0.1)')
    plt.xticks(rotation=30, ha='right')
    plt.ylabel('PSI')
    plt.title('Population Stability Index per feature')
    plt.legend()
    plt.tight_layout()
    plt.show()

## 6. Evaluate model performance

In [ ]:
import joblib
from monitoring.performance_tracker import compute_metrics

model = joblib.load('../model/baseline_model.pkl')

if 'target' in prod.columns:
    X = prod.drop(columns=['target'])
    y = prod['target']
    y_pred  = model.predict(X)
    y_proba = model.predict_proba(X)
    metrics = compute_metrics(y, y_pred, y_proba)
    pd.DataFrame([metrics])
else:
    print('No target column — skipping performance evaluation.')

## 7. Generate Evidently report

In [ ]:
from reports.generate_report import generate_evidently_report

generate_evidently_report(
    ref_path='../data/reference_data.csv',
    prod_path='../data/production_data.csv',
    output_path='../reports/monitoring_report.html',
)
print('Report written to ../reports/monitoring_report.html')